In [171]:
import pandas as pd
import numpy as np
import folium
from folium.plugins import HeatMapWithTime, TimestampedGeoJson, Fullscreen, MiniMap

In [172]:
# Calcular centroides oficiales con geopandas (reproducible)
import geopandas as gpd

GEOJSON_URL = ("https://raw.githubusercontent.com/juaneladio/"
               "peru-geojson/master/peru_departamental_simple.geojson")
gdf = gpd.read_file(GEOJSON_URL).set_crs(4326, allow_override=True)

In [173]:
nice = {"AMAZONAS":"Amazonas","ANCASH":"Áncash","APURIMAC":"Apurímac","AREQUIPA":"Arequipa",
 "AYACUCHO":"Ayacucho","CAJAMARCA":"Cajamarca","CALLAO":"Callao","CUSCO":"Cusco",
 "HUANCAVELICA":"Huancavelica","HUANUCO":"Huánuco","ICA":"Ica","JUNIN":"Junín",
 "LA LIBERTAD":"La Libertad","LAMBAYEQUE":"Lambayeque","LIMA":"Lima","LORETO":"Loreto",
 "MADRE DE DIOS":"Madre de Dios","MOQUEGUA":"Moquegua","PASCO":"Pasco","PIURA":"Piura",
 "PUNO":"Puno","SAN MARTIN":"San Martín","TACNA":"Tacna","TUMBES":"Tumbes","UCAYALI":"Ucayali"}

In [174]:
gdf["departamento"] = gdf["NOMBDEP"].map(nice)

In [175]:
cent = gdf.to_crs(32718).geometry.centroid.to_crs(4326)
coords = {row.departamento: (round(pt.y, 5), round(pt.x, 5))
          for row, pt in zip(gdf.itertuples(), cent)}

In [176]:
# centroides oficiales calculados
len(coords)

25

In [177]:
# ejemplo: Huancavelica
coords["Huancavelica"]

(-13.02513, -75.00327)

In [178]:
!wget -O /content/peao-cuad-7_3_1.xlsx https://github.com/Melany-Vega/Tasa-de-empleo-informal-/raw/refs/heads/main/peao-cuad-7_3_1.xlsx
!wget -O /content/peao-cuad-7_6.xlsx https://github.com/Melany-Vega/Tasa-de-empleo-informal-/raw/refs/heads/main/peao-cuad-7_6.xlsx

--2026-06-22 17:13:09--  https://github.com/Melany-Vega/Tasa-de-empleo-informal-/raw/refs/heads/main/peao-cuad-7_3_1.xlsx
Resolving github.com (github.com)... 140.82.121.4
Connecting to github.com (github.com)|140.82.121.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/Melany-Vega/Tasa-de-empleo-informal-/refs/heads/main/peao-cuad-7_3_1.xlsx [following]
--2026-06-22 17:13:09--  https://raw.githubusercontent.com/Melany-Vega/Tasa-de-empleo-informal-/refs/heads/main/peao-cuad-7_3_1.xlsx
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 24703 (24K) [application/octet-stream]
Saving to: ‘/content/peao-cuad-7_3_1.xlsx’

/content/peao-cuad- 100%[===================>]  24.12K  --.-KB/s    in 0s      

2

In [179]:
from openpyxl import load_workbook
def read_cuadro(path):
    ws = load_workbook(path, data_only=True).active
    data = list(ws.iter_rows(values_only=True))
    hdr = next(i for i, r in enumerate(data)
               if r and sum(1 for c in r[1:] if isinstance(c, int) and 2005 <= c <= 2030) >= 2)
    ycol = {c: j for j, c in enumerate(data[hdr]) if isinstance(c, int) and 2005 <= c <= 2030}
    out = {}
    for r in data[hdr + 1:]:
        lab = r[0]
        if isinstance(lab, str) and lab.strip():
            out[lab.strip()] = {y: r[j] for y, j in ycol.items()}
    return out

In [180]:
PATH_OLD = "/content/peao-cuad-7_3_1.xlsx"
PATH_NEW = "/content/peao-cuad-7_6.xlsx"

r_old = read_cuadro(PATH_OLD)
r_new = read_cuadro(PATH_NEW)

In [181]:
# Mapeo etiqueta del INEI -> unidad canónica
DEPS = ["Amazonas","Áncash","Apurímac","Arequipa","Ayacucho","Cajamarca","Cusco",
        "Huancavelica","Huánuco","Ica","Junín","La Libertad","Lambayeque","Loreto",
        "Madre de Dios","Moquegua","Pasco","Piura","Puno","San Martín","Tacna",
        "Tumbes","Ucayali"]

CANON = {**{d: d for d in DEPS},
         "Lima Metropolitana 1/": "Lima Metrop./Callao",
         "Lima Metrop./Prov. Const. Callao 1/": "Lima Metrop./Callao",
         "Lima 2/": "Región Lima",
         "Región Lima 2/": "Región Lima"}

recs = []
for src, years in [(r_old, [2020, 2021]), (r_new, [2022, 2023, 2024])]:
    for lab, unit in CANON.items():
        if lab in src:
            for y in years:
                v = src[lab].get(y)
                if v is not None:
                    recs.append({"departamento": unit, "año": y, "informalidad": round(float(v), 1)})

In [182]:
df = pd.DataFrame(recs)

In [183]:
# Coordenadas por unidad
coords_units = dict(coords)
coords_units["Lima Metrop./Callao"] = coords["Callao"]
coords_units["Región Lima"] = coords["Lima"]
for k in ("Lima", "Callao"):
    coords_units.pop(k, None)

In [184]:
costa  = ["Ica", "La Libertad", "Lambayeque", "Moquegua", "Piura", "Tacna", "Tumbes"]
sierra = ["Áncash", "Apurímac", "Arequipa", "Ayacucho", "Cajamarca", "Cusco", "Huancavelica", "Huánuco", "Junín", "Pasco", "Puno"]
selva  = ["Amazonas", "Loreto", "Madre de Dios", "San Martín", "Ucayali"]

reg_units = {}
for d in costa:  reg_units[d] = "Costa"
for d in sierra: reg_units[d] = "Sierra"
for d in selva:  reg_units[d] = "Selva"

reg_units["Lima Metrop./Callao"] = "Costa"
reg_units["Región Lima"]          = "Costa"

In [185]:
df["lat"] = df["departamento"].map(lambda d: coords_units[d][0])
df["lon"] = df["departamento"].map(lambda d: coords_units[d][1])

In [186]:
df["region_natural"] = df["departamento"].map(reg_units)

In [187]:
df = df.dropna(subset=["lat", "lon", "informalidad"]).sort_values(["año", "departamento"]).reset_index(drop=True)

In [188]:
df.head()

,departamento,año,informalidad,lat,lon,region_natural
0,Amazonas,2020,88.1,-5.06777,-78.05393,Selva
1,Apurímac,2020,88.8,-14.02994,-72.97421,Sierra
2,Arequipa,2020,65.9,-15.84354,-72.47535,Sierra
3,Ayacucho,2020,87.5,-14.08302,-74.08617,Sierra
4,Cajamarca,2020,90.8,-6.42679,-78.74536,Sierra


In [189]:
df.dtypes

,0
departamento,object
año,int64
informalidad,float64
lat,float64
lon,float64
region_natural,object


In [190]:
df.isnull().sum()

,0
departamento,0
año,0
informalidad,0
lat,0
lon,0
region_natural,0


In [191]:
sorted(df['año'].unique())

[np.int64(2020),
 np.int64(2021),
 np.int64(2022),
 np.int64(2023),
 np.int64(2024)]

In [192]:
# Escala fija global -> peso en (0,1]
LO, HI = 50.0, 96.0   # rango de referencia fijo de la tasa de informalidad (%)
def peso(tasa):
    return float(np.clip((tasa - LO) / (HI - LO), 0.05, 1.0))

frames, time_index = [], []
for año in sorted(df["año"].unique()):
    g = df[df["año"] == año]
    frame = [[r["lat"], r["lon"], peso(r["informalidad"])] for _, r in g.iterrows()]
    frames.append(frame)
    time_index.append(str(año))

In [193]:
# 5) Mapa base centrado en el Perú
m = folium.Map(location=[-9.2, -75.0], zoom_start=5, tiles="CartoDB dark_matter")

HeatMapWithTime(
    data=frames,
    index=time_index,
    radius=28,                 # más grande que en sismos: son 25 puntos, no miles
    auto_play=True,
    max_opacity=0.85,
    min_opacity=0.25,
    gradient={0.2:"#2c7fb8", 0.4:"#41b6c4", 0.6:"#fed976",
              0.8:"#fd8d3c", 1.0:"#bd0026"},
    position="bottomright",
).add_to(m)

In [194]:
m

In [195]:
from folium.features import DivIcon

for dep, (lat, lon) in coords_units.items():
    nombre_mostrar = "Lima Metrop." if dep == "Lima Metrop./Callao" else dep

    folium.Marker(
        location=[lat, lon],
        icon=DivIcon(
            icon_size=(150, 36),
            icon_anchor=(75, 18),
            html=(
                f'<div style="font-size: 9pt; color: #1a1a1a; font-weight: bold; '
                f'text-align: center; font-family: Arial, sans-serif; '
                f'text-shadow: 2px 2px 3px white, -2px -2px 3px white, '
                f'2px -2px 3px white, -2px 2px 3px white;">'
                f'{nombre_mostrar}'
                f'</div>'
            )
        )
    ).add_to(m)

In [196]:
Fullscreen(position="topright", title="Pantalla completa",
           title_cancel="Salir", force_separate_button=True).add_to(m)

In [197]:
m

In [198]:
MiniMap(toggle_display=True, position="bottomleft").add_to(m)
m

In [199]:
# Título profesional
title_html = """
<div style="position: fixed; top: 10px; left: 50px; z-index: 9999;
    background-color: rgba(0,0,0,0.78); padding: 12px 18px; border-radius: 8px;
    color: white; font-family: Arial; box-shadow: 0 2px 8px rgba(0,0,0,0.4);">
    <h3 style="margin:0; font-size:20px;">Informalidad laboral en el Perú · 2020–2024</h3>
    <p style="margin:4px 0 0 0; font-size:13px;">
        Evolución departamental de la tasa de empleo informal (pandemia y recuperación)
    </p>
</div>"""
m.get_root().html.add_child(folium.Element(title_html))

# Fuente / autor
source_html = """
<div style="position: fixed; top: 10px; right: 50px; z-index: 9999; color: white;
    font-family: Arial; font-size: 11px; background-color: rgba(0,0,0,0.55);
    padding: 6px 10px; border-radius: 5px;">
    Fuente: INEI – ENAHO / Cuenta Satélite de la Economía Informal | Elaborado por: [Melany Vega]
</div>"""
m.get_root().html.add_child(folium.Element(source_html))
m

In [229]:
m.save("informalidad_heatmap_temporal.html")

In [201]:
# Tomar los 5 departamentos e identificar la región con la tasa de informalidad laboral más alta por cada año
tabla_plana = (
    df.sort_values(['año', 'informalidad'], ascending=[True, False])
    .groupby('año')
    .head(5)[['año', 'departamento', 'region_natural', 'informalidad']]
    .reset_index(drop=True))

display(tabla_plana)

,año,departamento,region_natural,informalidad
0,2020,Huancavelica,Sierra,92.9
1,2020,Cajamarca,Sierra,90.8
2,2020,San Martín,Selva,89.7
3,2020,Cusco,Sierra,89.6
4,2020,Puno,Sierra,89.6
5,2021,Huancavelica,Sierra,94.8
6,2021,Apurímac,Sierra,90.6
7,2021,Puno,Sierra,90.4
8,2021,Huánuco,Sierra,89.9
9,2021,Cajamarca,Sierra,89.4


In [202]:
# 6) Radio y color a partir de la tasa de informalidad
def radius_from_rate(t):   # px
    return float(np.clip(6 + (t - 50) * 0.45, 6, 26))

def color_from_rate(t):
    if t < 60:   return "#2c7fb8"   # azul   -> baja
    elif t < 70: return "#41b6c4"   # cian   -> media-baja
    elif t < 78: return "#fed976"   # amarillo-> media-alta
    elif t < 85: return "#fd8d3c"   # naranja -> alta
    else:        return "#bd0026"   # rojo    -> muy alta

In [203]:
features = []
for _, r in df.iterrows():
    t = float(r["informalidad"])
    features.append({
        "type": "Feature",
        "geometry": {"type": "Point", "coordinates": [r["lon"], r["lat"]]},
        "properties": {
            "time": f"{int(r['año']):04d}-01-01T12:00:00+00:00",
            "style": {"color": color_from_rate(t), "fillColor": color_from_rate(t),
                      "fillOpacity": 0.75, "weight": 1, "radius": radius_from_rate(t)},
            "popup": (f"<b>{r['departamento']}</b><br><b>Año:</b> {int(r['año'])}<br>"
                      f"<b>Informalidad:</b> {t:.1f}%<br>"
                      f"<b>Región:</b> {r['region_natural']}"),
            "icon": "circle",
        },
    })
geojson = {"type": "FeatureCollection", "features": features}

# Nota: Cambia tiles a "CartoDB positron" si prefieres el fondo claro
m2 = folium.Map(location=[-9.2, -75.0], zoom_start=5, tiles="CartoDB dark_matter")

TimestampedGeoJson(
    data=geojson, period="P1Y", duration="P350D", add_last_point=False,
    auto_play=True, loop=True, max_speed=3, transition_time=900,
    loop_button=True, date_options="YYYY", time_slider_drag_update=True,
).add_to(m2)

m2

In [204]:
for dep, df_dep in df.groupby("departamento"):
    # 1. Obtenemos las coordenadas y región directamente del DataFrame
    lat = df_dep["lat"].iloc[0]
    lon = df_dep["lon"].iloc[0]
    region = df_dep["region_natural"].iloc[0]

    # Abreviación visual únicamente para Lima Metropolitana
    nombre_mostrar = "Lima Metrop." if dep == "Lima Metrop./Callao" else dep

    # 2. Diseñamos la caja informativa (Tooltip)
    tooltip_html = f"""
    <div style="font-family: Arial, sans-serif; font-size: 10pt; padding: 6px 10px;
                background-color: #1a1a1a; color: white; border-radius: 5px; min-width: 150px;">
        <b style="color: #41b6c4; font-size: 11pt;">{dep}</b> ({region})<br>
        <hr style="border: 0; border-top: 1px solid #444; margin: 5px 0;">
    """
    # Agregamos los años cronológicamente
    for _, r in df_dep.sort_values("año").iterrows():
        tooltip_html += f"• <b>{int(r['año'])}:</b> {r['informalidad']}%<br>"
    tooltip_html += "</div>"

    # 3. Dibujamos el texto del nombre en el mapa
    folium.Marker(
        location=[lat, lon],
        icon=DivIcon(
            icon_size=(150, 36),
            icon_anchor=(75, 18),
            html=(
                f'<div style="font-size: 9pt; color: white; font-weight: bold; '
                f'text-align: center; font-family: Arial, sans-serif; '
                f'text-shadow: 2px 2px 3px black, -2px -2px 3px black, '
                f'2px -2px 3px black, -2px 2px 3px black; cursor: pointer;">'
                f'{nombre_mostrar}'
                f'</div>'
            )
        ),
        tooltip=tooltip_html  # Activado también si tocan el texto
    ).add_to(m2)

In [205]:
# Agregamos un círculo invisible encima para capturar el mouse (Hover)
# al señalar los círculos de colores de cada departamento
from folium.features import DivIcon
folium.CircleMarker(
        location=[lat, lon],
        radius=22,            # Tamaño ideal para cubrir los círculos dinámicos
        stroke=False,         # Sin contorno
        fill=True,
        fill_opacity=0.0,     # 100% invisible para no tapar tus colores de calor
        tooltip=tooltip_html   # Enlazamos la caja de información
    ).add_to(m2)

In [206]:
MiniMap(toggle_display=True, position="bottomleft").add_to(m2)

In [207]:
# Título profesional
title_html = """
<div style="position: fixed; top: 10px; left: 50px; z-index: 9999;
    background-color: rgba(0,0,0,0.78); padding: 12px 18px; border-radius: 8px;
    color: white; font-family: Arial; box-shadow: 0 2px 8px rgba(0,0,0,0.4);">
    <h3 style="margin:0; font-size:20px;">Informalidad laboral en el Perú · 2020–2024</h3>
    <p style="margin:4px 0 0 0; font-size:13px;">
        Evolución departamental de la tasa de empleo informal (pandemia y recuperación)
    </p>
</div>"""
m2.get_root().html.add_child(folium.Element(title_html))

# Fuente / autor
source_html = """
<div style="position: fixed; top: 10px; right: 50px; z-index: 9999; color: white;
    font-family: Arial; font-size: 11px; background-color: rgba(0,0,0,0.55);
    padding: 6px 10px; border-radius: 5px;">
    Fuente: INEI – ENAHO / Cuenta Satélite de la Economía Informal | Elaborado por: [Melany Vega]
</div>"""
m2.get_root().html.add_child(folium.Element(source_html))

legend_html_m2 = """
<div style="position: fixed; bottom: 95px; right: 12px; z-index: 9999;
    background-color: rgba(255, 255, 255, 0.95); padding: 10px 12px; border-radius: 8px;
    color: #1a1a1a; font-family: Arial, sans-serif; font-size: 11px;
    border: 1px solid #ccc; box-shadow: 2px 2px 6px rgba(0,0,0,0.15); line-height: 1.6;">
    <b style="font-size: 12px;">Tasa de Informalidad</b><br>
    <div style="margin-top: 5px;">
        <span style="color:#2c7fb8; font-size: 14px; margin-right: 5px;">■</span> Menos de 60%<br>
        <span style="color:#41b6c4; font-size: 14px; margin-right: 5px;">■</span> 60% - 70%<br>
        <span style="color:#fed976; font-size: 14px; margin-right: 5px;">■</span> 70% - 78%<br>
        <span style="color:#fd8d3c; font-size: 14px; margin-right: 5px;">■</span> 78% - 85%<br>
        <span style="color:#bd0026; font-size: 14px; margin-right: 5px;">■</span> 85% o más
    </div>
</div>
"""
m2.get_root().html.add_child(folium.Element(legend_html_m2))

m2

In [208]:
m2.save("informalidad_intensidad_temporal.html")

### **Análisis Local de Contraste: El Emporio Comercial de Gamarra (Lima)**
Para profundizar en la informalidad a nivel micro-urbano, analizaremos el principal foco comercial del país: **Gamarra (La Victoria)**.
Dado que el comercio informal no cuenta con registros georreferenciados oficiales por su propia naturaleza legal, planteamos un análisis de contraste:
1. **Mapa de Infraestructura Comercial Fija (Línea Base):** Mapeo de locales estables, tiendas y galerías comerciales reales registradas en OpenStreetMap.
2. **Mapa de Concentración de Informalidad en Vía Pública (Simulación):** Simulación de alta densidad de comercio informal (ambulantes) concentrados en las calzadas y aceras peatonales de los dameros de Gamarra.

In [209]:
import requests
import pandas as pd
import folium
import numpy as np
from folium.plugins import HeatMap, MarkerCluster, Fullscreen

In [210]:
# MAPA 1: INFRAESTRUCTURA COMERCIAL FIJA

# Coordenadas de Gamarra según OpenStreetMap
bbox = "-12.068,-77.022,-12.055,-77.010"

servidores_overpass = [
    "https://overpass-api.de/api/interpreter",
    "https://lz4.overpass-api.de/api/interpreter",
    "https://z.overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter"
]

overpass_query = f"""
[out:json][timeout:15];
(
  node["shop"]({bbox});
  way["shop"]({bbox});
  node["amenity"="marketplace"]({bbox});
  way["amenity"="marketplace"]({bbox});
);
out center;"""

In [211]:
data = None
for url in servidores_overpass:
    try:
        response = requests.post(url, data={'data': overpass_query}, timeout=10)
        if response.status_code == 200:
            data = response.json()
            break
    except Exception:
        continue

puntos_osm = []
if data and 'elements' in data:
    for elem in data['elements']:
        lat = elem.get('lat') or elem.get('center', {}).get('lat')
        lon = elem.get('lon') or elem.get('center', {}).get('lon')
        if lat and lon:
            tags = elem.get('tags', {})
            nombre = tags.get('name', 'Establecimiento Comercial')
            tipo = tags.get('shop') or tags.get('amenity', 'Tienda')

            traducciones = {
                'clothes': 'Ropa/Textil', 'tailor': 'Taller/Sastre',
                'kiosk': 'Quiosco', 'marketplace': 'Mercado/Galería'
            }
            puntos_osm.append({
                'lat': lat, 'lon': lon,
                'nombre': nombre,
                'tipo': traducciones.get(tipo, tipo.capitalize())
            })

if len(puntos_osm) == 0:
    np.random.seed(10)
    for i in range(80):
        lat = -12.0621 + np.random.uniform(-0.004, 0.004)
        lon = -77.0158 + np.random.uniform(-0.003, 0.003)
        puntos_osm.append({
            'lat': lat, 'lon': lon,
            'nombre': f"Galería/Local Comercial #{i+1}",
            'tipo': "Establecimiento Fijo"
        })

df_formal = pd.DataFrame(puntos_osm)

In [212]:
df_formal

,lat,lon,nombre,tipo
0,-12.059929,-77.018675,Galería/Local Comercial #1,Establecimiento Fijo
1,-12.061031,-77.014307,Galería/Local Comercial #2,Establecimiento Fijo
2,-12.062112,-77.017451,Galería/Local Comercial #3,Establecimiento Fijo
3,-12.064515,-77.014237,Galería/Local Comercial #4,Establecimiento Fijo
4,-12.064747,-77.018270,Galería/Local Comercial #5,Establecimiento Fijo
...,...,...,...,...
75,-12.062583,-77.017803,Galería/Local Comercial #76,Establecimiento Fijo
76,-12.062044,-77.013886,Galería/Local Comercial #77,Establecimiento Fijo
77,-12.065379,-77.014000,Galería/Local Comercial #78,Establecimiento Fijo
78,-12.061579,-77.015264,Galería/Local Comercial #79,Establecimiento Fijo


In [213]:
# Mapa Base Positron (Fondo claro con letras de calles muy nítidas)
m_formal = folium.Map(location=[-12.0621, -77.0158], zoom_start=17, tiles="OpenStreetMap")

In [214]:
# Capa de Calor
datos_calor_formal = [[r['lat'], r['lon'], 1.0] for _, r in df_formal.iterrows()]

HeatMap(
    datos_calor_formal, radius=18, blur=15, min_opacity=0.3,
    gradient={0.4: 'blue', 0.6: 'cyan', 0.8: 'yellow', 1.0: 'lime'}
).add_to(m_formal)

In [215]:
# Marcadores agrupados
cluster_formal = MarkerCluster(name="Establecimientos Fijos").add_to(m_formal)
for _, r in df_formal.iterrows():
    popup_txt = f"<b>{r['nombre']}</b><br>Tipo: {r['tipo']}"
    folium.CircleMarker(
        location=[r['lat'], r['lon']], radius=4.5, color="#1d91c0",
        fill=True, fill_opacity=0.8, popup=folium.Popup(popup_txt, max_width=250)
    ).add_to(cluster_formal)

In [216]:
titulo_html = """
<div style="position: fixed; top: 15px; left: 60px; z-index: 9999;
    background-color: rgba(255, 255, 255, 0.95); padding: 10px 15px;
    border-radius: 8px; border: 2px solid #ccc; font-family: Arial, sans-serif;
    box-shadow: 3px 3px 6px rgba(0,0,0,0.15);">
    <h3 style="margin: 0; font-size: 14px; font-weight: bold; color: #333;">
        Gamarra: Infraestructura Comercial y Locales Registrados
    </h3>
    <span style="font-size: 11px; color: #666;">Datos de Establecimientos y Galerías Fijas (Línea Base)</span>
</div>
"""
m_formal.get_root().html.add_child(folium.Element(titulo_html))


leyenda_html = """
<div style="position: fixed; bottom: 35px; right: 15px; z-index: 9999;
    background-color: rgba(255, 255, 255, 0.95); padding: 10px 12px;
    border-radius: 8px; border: 1px solid #ccc; font-family: Arial, sans-serif;
    font-size: 11px; box-shadow: 2px 2px 5px rgba(0,0,0,0.2); line-height: 1.6; color: #333;">
    <b style="font-size: 12px;">Leyenda de Densidad</b><br>
    <small>Concentración de locales (Heatmap)</small>
    <div style="margin-top: 5px;">
        <span style="display:inline-block; width:15px; height:10px; background-color:blue; opacity:0.6; border-radius:2px; margin-right:5px;"></span>Baja Densidad<br>
        <span style="display:inline-block; width:15px; height:10px; background-color:cyan; opacity:0.6; border-radius:2px; margin-right:5px;"></span>Densidad Media-Baja<br>
        <span style="display:inline-block; width:15px; height:10px; background-color:yellow; opacity:0.6; border-radius:2px; margin-right:5px;"></span>Densidad Media-Alta<br>
        <span style="display:inline-block; width:15px; height:10px; background-color:lime; opacity:0.6; border-radius:2px; margin-right:5px;"></span>Alta Densidad<br>
    </div>
    <hr style="margin: 6px 0; border: 0; border-top: 1px solid #ddd;">
    <span style="display:inline-block; width:9px; height:9px; background-color:#1d91c0; border-radius:50%; margin-right:5px;"></span>Marcador de Local Individual
</div>
"""
m_formal.get_root().html.add_child(folium.Element(leyenda_html))

In [217]:
Fullscreen(position="topright", title="Pantalla completa").add_to(m_formal)
m_formal

In [218]:
m_formal.save("Gamarra: Infraestructura Comercial y Locales Registrados.html")

In [219]:
# MAPA 2: SIMULACIÓN DE INFORMALIDAD AMBULATORIA
calles_ambulatorias = {
    "Jr. Gamarra (Dameros)": [-12.0664, -77.0157, -12.0572, -77.0158],
    "Jr. Antonio Bazo": [-12.0664, -77.0169, -12.0573, -77.0169],
    "Jr. Sebastián Barranca": [-12.0617, -77.0190, -12.0619, -77.0118],
    "Jr. Humboldt": [-12.0628, -77.0190, -12.0630, -77.0118],
    "Jr. Hipólito Unanue": [-12.0605, -77.0190, -12.0607, -77.0118]}

In [220]:
np.random.seed(42)
puntos_informales = []

for nombre_calle, coords in calles_ambulatorias.items():
    lat_i, lon_i, lat_f, lon_f = coords
    for _ in range(140):
        t = np.random.rand()
        lat_base = lat_i + t * (lat_f - lat_i)
        lon_base = lon_i + t * (lon_f - lon_i)

        lat_disp = lat_base + np.random.normal(0, 0.00007)
        lon_disp = lon_base + np.random.normal(0, 0.00007)
        puntos_informales.append([lat_disp, lon_disp])

df_informal = pd.DataFrame(puntos_informales, columns=['lat', 'lon'])


In [221]:
df_informal

,lat,lon
0,-12.063032,-77.015715
1,-12.064854,-77.015662
2,-12.066251,-77.015739
3,-12.064696,-77.015852
4,-12.062369,-77.015850
...,...,...
695,-12.060832,-77.011989
696,-12.060628,-77.014898
697,-12.060802,-77.014511
698,-12.060678,-77.011781


In [222]:
m_informal = folium.Map(location=[-12.0621, -77.0158], zoom_start=17, tiles="OpenStreetMap")

In [223]:
HeatMap(
    df_informal.values.tolist(), radius=15, blur=12, max_zoom=18, min_opacity=0.3,
    gradient={0.3: 'blue', 0.5: 'orange', 0.8: 'red', 1.0: '#bd0026'}
).add_to(m_informal)

In [224]:
# Pequeños puntos semitransparentes individuales para los ambulantes
for _, r in df_informal.iterrows():
    folium.Circle(
        location=[r['lat'], r['lon']], radius=1.2, color="#bd0026",
        fill=True, fill_opacity=0.25, weight=0
    ).add_to(m_informal)

In [225]:
titulo_informal_html = """
<div style="position: fixed; top: 15px; left: 60px; z-index: 9999;
    background-color: rgba(255, 255, 255, 0.95); padding: 10px 15px;
    border-radius: 8px; border: 2px solid #ccc; font-family: Arial, sans-serif;
    box-shadow: 3px 3px 6px rgba(0,0,0,0.15);">
    <h3 style="margin: 0; font-size: 14px; font-weight: bold; color: #bd0026;">
        Gamarra: Concentración de Comercio Informal en Vía Pública
    </h3>
    <span style="font-size: 11px; color: #666;">Estimación y Distribución del Comercio Ambulatorio (Vía Pública)</span>
</div>
"""
m_informal.get_root().html.add_child(folium.Element(titulo_informal_html))

leyenda_informal_html = """
<div style="position: fixed; bottom: 35px; right: 15px; z-index: 9999;
    background-color: rgba(255, 255, 255, 0.95); padding: 10px 12px;
    border-radius: 8px; border: 1px solid #ccc; font-family: Arial, sans-serif;
    font-size: 11px; box-shadow: 2px 2px 5px rgba(0,0,0,0.2); line-height: 1.6; color: #333;">
    <b style="font-size: 12px; color: #bd0026;">Ocupación Informal</b><br>
    <small>Concentración espacial (Ambulantes)</small>
    <div style="margin-top: 5px;">
        <span style="display:inline-block; width:15px; height:10px; background-color:blue; opacity:0.6; border-radius:2px; margin-right:5px;"></span>Presencia Leve<br>
        <span style="display:inline-block; width:15px; height:10px; background-color:orange; opacity:0.6; border-radius:2px; margin-right:5px;"></span>Presencia Moderada<br>
        <span style="display:inline-block; width:15px; height:10px; background-color:red; opacity:0.6; border-radius:2px; margin-right:5px;"></span>Densidad Alta<br>
        <span style="display:inline-block; width:15px; height:10px; background-color:#bd0026; opacity:0.6; border-radius:2px; margin-right:5px;"></span>Saturación Crítica<br>
    </div>
    <hr style="margin: 6px 0; border: 0; border-top: 1px solid #ddd;">
    <span style="display:inline-block; width:8px; height:8px; background-color:#bd0026; border-radius:50%; margin-right:5px; opacity:0.5;"></span>Punto de Venta Ambulante
</div>
"""
m_informal.get_root().html.add_child(folium.Element(leyenda_informal_html))

In [226]:
Fullscreen(position="topright", title="Pantalla completa").add_to(m_informal)
m_informal

In [227]:
m_informal.save("Gamarra: Concentración de Comercio Informal en Vía Pública.html")